# 02 — Train

This notebook trains a forecast-error model for every station in your
bounding box, for every forecast hour and meteorological variable in your
`config.yaml`.

**Prerequisites**: run `01_setup_data.ipynb` first so that all parquets
and the station-cluster lookup are ready.

**Steps**
1. Load config and review the train / val / test split.
2. Visualise the chronological split as a timeline bar.
3. Train the model — for each `(fh, metvar)` pair, one model is saved
   per station in `./output/models/`.
4. Optionally plot per-epoch loss curves (accumulated during training).

In [ ]:
import sys, os

REPO_ROOT = os.path.abspath("../..")
APP_ROOT  = os.path.abspath("..")
for p in (REPO_ROOT, APP_ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(APP_ROOT)
print("Working directory:", os.getcwd())

In [ ]:
from app.utils.config_loader import load_config

cfg = load_config()
cfg.make_output_dirs()

print("Model type       :", cfg.model)
print("Forecast hours   :", cfg.data.forecast_hours)
print("Met variables    :", cfg.data.metvars)
print("Epochs           :", cfg.training.epochs)
print("Batch size       :", cfg.training.batch_size)
print("Learning rate    :", cfg.training.learning_rate)
print("Sequence length  :", cfg.training.sequence_length)
print("Device id        :", cfg.training.device_id)

## Step 1 — Visualise the train / val / test split

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import pandas as pd

t = cfg.training
dates = [
    ("Train",      t.start_date,   t.train_end,  "#2a9d8f"),
    ("Validation", t.train_end,    t.val_end,    "#e9c46a"),
    ("Test",       t.val_end,      t.test_end,   "#e76f51"),
]

fig, ax = plt.subplots(figsize=(12, 1.5))
for label, d_start, d_end, colour in dates:
    ax.barh(
        0,
        (d_end - d_start).days,
        left=mdates.date2num(d_start),
        height=0.6,
        color=colour,
        label=f"{label}  ({d_start} → {d_end})",
    )

ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.yaxis.set_visible(False)
ax.set_title("Chronological data split")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=3, frameon=False)
plt.tight_layout()
plt.savefig(f"{cfg.output.results_dir}/split_timeline.png", dpi=150)
plt.show()

print(f"Training  : {t.start_date}  →  {t.train_end}")
print(f"Validation: {t.train_end}  →  {t.val_end}")
print(f"Test      : {t.val_end}  →  {t.test_end}")

## Step 2 — Set up the training environment

The cell below:
- sets `LSTM_MODEL_DIR` so trained weights land in `./output/models/`
- patches the data loaders to read from `./output/parquets/` instead of the
  hard-coded server paths
- writes a station CSV that the training engine uses to iterate over stations
- patches the station-cluster lookup to use your ASOS stations

In [ ]:
import torch
from datetime import datetime
from app.utils.engine_bridge import (
    prepare_env,
    make_station_csv,
    patch_lookup_parquet,
)

# Set env vars (must happen before importing the engine modules).
prepare_env(cfg)

# Write a NYSM-compatible station CSV for the training engines.
station_csv_path = make_station_csv(cfg)
print("Station CSV written to:", station_csv_path)

# Redirect the station-cluster lookup to the app's parquet.
patch_lookup_parquet(cfg)

device = torch.device(f"cuda:{cfg.training.device_id}"
                      if torch.cuda.is_available() else "cpu")
print("Training on:", device)

## Step 3 — Train

The outer loop iterates over each forecast hour; for each hour the engine
trains one model per `(station, metvar)` pair.  Loss curves are stored in
`loss_history` for plotting after training.

In [ ]:
from datetime import timedelta
from app.utils.engine_bridge import patch_data_loaders, patch_station_csv

# Convert config dates to datetime for the engine
train_start = datetime(
    cfg.training.start_date.year,
    cfg.training.start_date.month,
    cfg.training.start_date.day,
)
train_end = datetime(
    cfg.training.train_end.year,
    cfg.training.train_end.month,
    cfg.training.train_end.day,
)

# Synthetic climate division name used by the engine to find bbox stations.
CLIM_DIV = "bbox_stations"

# Pick the right engine module based on cfg.model.
engine_module_map = {
    "lstm":   "src.engine_lstm_training",
    "bnn":    "src.engine_bnn_training",
    "hybrid": "src.engine_hybrid_training",
}
engine_module_name = engine_module_map.get(cfg.model)
if engine_module_name is None:
    raise ValueError(f"Unknown model type '{cfg.model}'. Choose lstm | bnn | hybrid.")

loss_history = {}  # {(fh, metvar): [(epoch, train_loss, val_loss), ...]}

with patch_data_loaders(cfg), patch_station_csv(cfg) as clim_div:
    import importlib
    engine = importlib.import_module(engine_module_name)

    for fh in cfg.data.forecast_hours:
        print(f"\n{'='*60}")
        print(f"  Forecast hour: fh={fh}")
        print(f"{'='*60}")

        # Load HRRR data for this forecast hour via the patched loader.
        import src.model_data.hrrr_data as hrrr_data
        hrrr_df = hrrr_data.read_hrrr_data(str(fh).zfill(2), train_start.year)

        engine.main(
            start_time=train_start,
            end_time=train_end,
            batch_size=cfg.training.batch_size,
            num_layers=cfg.training.num_layers,
            epochs=cfg.training.epochs,
            weight_decay=cfg.training.weight_decay,
            fh=fh,
            clim_div=clim_div,
            device=device,
            hrrr_df=hrrr_df,
            sequence_length=cfg.training.sequence_length,
            learning_rate=cfg.training.learning_rate,
        )

print("\nTraining complete.  Weights saved to:", cfg.output.models_dir)

## Step 4 — Review saved model files

In [ ]:
from pathlib import Path

model_files = sorted(Path(cfg.output.models_dir).glob("*.pth"))
print(f"{len(model_files)} weight files written:")
for f in model_files[:20]:
    print(" ", f.name)
if len(model_files) > 20:
    print(f"  … and {len(model_files) - 20} more.")

## Next steps

- **`03_evaluate.ipynb`** — run on the test split and compute error metrics.
- **`04_inference.ipynb`** — run on a specific `now` timestamp to produce
  real-time predictions.